In [1]:
from netCDF4 import Dataset

In [2]:
from AutoEncoder_Experiment import *
import sys, jax
import training_functions as tf
import jax.numpy as jnp
from datetime import datetime
jax.config.update("jax_enable_x64", True)


from training_functions import *

In [3]:
#JSON is arg
json_fn = 'Sigmoids_Dropout3.json'
#To be added to json
num_rmsd_epochs = 100 #Initial Run
nm_cutoff = 0.1 #target rmsd in nm
rmsd_cutoff = 1500 #num epochs to force training to summation
num_move_ave = 50 #number of previous points for moving average calculations
potential_threshold = 1e-3 #Target Potential NSD tolerance
cutoff_epoch = 30000 #maximum number of epochs of training

In [4]:
#Init Experiment
experiment = AutoEncoder_Experiment(json_fn)

(8000, 306) (2000, 306)
### MODEL TYPE = AE_Sigmoid_Dropout ###
Sigmoid_Dropout_AutoEncoder(
    # attributes
    input_size = 306
    hidden_layers = [306, 306, 306]
    n_latents = 7
    dropout_rates = [0.3, 0.3, 0.3]
)
INITIALIZATION COMPLETE


In [5]:
start = datetime.now()

In [6]:
experiment.train_nepochs(tf.rmsd_rng_step, 25)

2023-11-26 06:01:19.638388: E external/xla/xla/service/slow_operation_alarm.cc:65] Constant folding an instruction is taking > 1s:

  %slice.85 = s32[202213,1]{1,0} slice(s32[202213,2]{1,0} %constant.59), slice={[0:202213], [0:1]}, metadata={op_name="jit(scaled_pot_enr_diff)/jit(main)/vmap(jit(compute_fn))/slice[start_indices=(0, 0) limit_indices=(202213, 1) strides=None]" source_file="/media/volume/sdb/githubs/Deep-MMS/jax_amber3.py" source_line=372}

This isn't necessarily a bug; constant-folding is inherently a trade-off between compilation time and speed at runtime. XLA has some guards that attempt to keep constant folding from taking too long, but fundamentally you'll always be able to come up with an input program that takes a long time.

If you'd like to file a bug, run with envvar XLA_FLAGS=--xla_dump_to=/tmp/foo and attach the results.
2023-11-26 06:01:19.734896: E external/xla/xla/service/slow_operation_alarm.cc:133] The operation took 1.096601401s
Constant folding an instruc

epoch 0 atom_rmsd_nm 1.1606E+00 1.1605E+00 torsional 5.7807E-01 5.7764E-01 dPotEnr NAN NAN Summation NAN NAN B=0.0000E+00 L=0.0000E+00
epoch 1 atom_rmsd_nm 1.1486E+00 1.1485E+00 torsional 5.7826E-01 5.7784E-01 dPotEnr NAN NAN Summation NAN NAN B=0.0000E+00 L=0.0000E+00
epoch 2 atom_rmsd_nm 1.1366E+00 1.1365E+00 torsional 5.7854E-01 5.7812E-01 dPotEnr NAN NAN Summation NAN NAN B=0.0000E+00 L=0.0000E+00
epoch 3 atom_rmsd_nm 1.1248E+00 1.1247E+00 torsional 5.7891E-01 5.7849E-01 dPotEnr NAN NAN Summation NAN NAN B=0.0000E+00 L=0.0000E+00
epoch 4 atom_rmsd_nm 1.1130E+00 1.1129E+00 torsional 5.7924E-01 5.7883E-01 dPotEnr NAN NAN Summation NAN NAN B=0.0000E+00 L=0.0000E+00
epoch 5 atom_rmsd_nm 1.1013E+00 1.1012E+00 torsional 5.7949E-01 5.7909E-01 dPotEnr NAN NAN Summation NAN NAN B=0.0000E+00 L=0.0000E+00
epoch 6 atom_rmsd_nm 1.0897E+00 1.0896E+00 torsional 5.7982E-01 5.7943E-01 dPotEnr NAN NAN Summation NAN NAN B=0.0000E+00 L=0.0000E+00
epoch 7 atom_rmsd_nm 1.0782E+00 1.0781E+00 torsional 5.

In [7]:
experiment.train_scaling_coef(tf.structural_rng_step, 100, 0)

epoch 25 atom_rmsd_nm NAN NAN torsional NAN NAN dPotEnr NAN NAN Summation NAN NAN B=0.0000E+00 L=0.0000E+00
epoch 26 atom_rmsd_nm NAN NAN torsional NAN NAN dPotEnr NAN NAN Summation NAN NAN B=0.0000E+00 L=0.0000E+00
epoch 27 atom_rmsd_nm NAN NAN torsional NAN NAN dPotEnr NAN NAN Summation NAN NAN B=0.0000E+00 L=0.0000E+00


KeyboardInterrupt: 

In [ ]:
experiment.train_scaling_torsional(400, freq=5)

In [ ]:
end_scaling_epoch = experiment.train_scaling_potential(cutoff_epoch, freq=5)
#freq=number of epochs before reevaluate pot rmsd ratio,  this should be low as the dataset is large (many batches per epoch)
print('END SCALING POTENTIAL')
walltime = datetime.now() - start
print(f'Walltime per epoch was {walltime}')
experiment.save_loss_data()
experiment.write_model_to_ckpt()

In [ ]:
experiment.train_summation(1e-1, 10, cutoff_epoch)
print('END TRAINING')
walltime = datetime.now() - start
print(f'Walltime was {walltime}')
#experiment.save_loss_data()
#experiment.write_model_to_ckpt()

In [ ]:
rng_init = jax.random.PRNGKey(54)
rng, key = jax.random.split(rng_init)
recon, latents = experiment.state.apply_fn({'params':experiment.state.params}, experiment.test_data, rng)

In [ ]:
for i in range(latents.shape[-1]):
    plt.clf()
    plt.title(f'Latent {i}')
    _ = plt.hist(latents[:, i], bins=100)
    plt.show()

In [ ]:
from sklearn.mixture import GaussianMixture

In [ ]:
ics = []
for i in range(1, 31):
    X = np.array(latents)
    MM = GaussianMixture(n_components=i).fit(X)
    ics.append((i, MM.aic(X), MM.bic(X)))
ics = np.array(ics)
plt.clf()
_ = plt.plot(ics[:, 0], ics[:, 1])
_ = plt.plot(ics[:, 0], ics[:, 2])
plt.legend(('Akaike Info Criterion', 'Bayes Info Criterion'))
plt.xlabel('Num Components')
plt.show()

In [ ]:
X = np.array(latents)
MM = GaussianMixture(n_components=1).fit(X) #chosen based on above graph
samples = MM.sample(1800)[0]

In [ ]:
for i in range(samples.shape[-1]):
    plt.clf()
    _ = plt.hist(latents[:,i], bins=100, histtype='step', color='g')
    _ = plt.hist(samples[:,i], bins=100, histtype='step', color='b')
    plt.legend(('Reconstructed from Input', 'Sampled from Mixture Model'))
    plt.title(f'Latent {i+1}')
    plt.show()

In [ ]:
n_L = latents.shape[-1]
fig, axs = plt.subplots(n_L, n_L, figsize=(15, 10), sharex='col', sharey='row')
for i in range(n_L):
    for j in range(n_L):
        axs[i,j].scatter(latents[:,j], latents[:,i])
fig.savefig('7L.png', dpi=600)

In [ ]:
from jax_amber2 import *
gas_fun = get_amber_gas_energy_function('Simulation/1crn_H.prmtop')

In [ ]:
plt.clf()

recon_energies = gas_fun(recon)

decoded_samples = experiment.model.apply({'params': experiment.state.params}, samples, rng, method=experiment.model.decode)
sampled_energies = gas_fun(decoded_samples)

org_energies = gas_fun(experiment.test_data)

threshold = 1e3
num_excluded_gmm = len(sampled_energies) - len(sampled_energies[sampled_energies<threshold])
num_excluded_nn  = len(recon_energies) - len(recon_energies[recon_energies<threshold])
print(f'Excluding Outliers GMM - {num_excluded_gmm} RECON - {num_excluded_nn}')

_ = plt.hist(org_energies[org_energies < threshold], bins=100, histtype='step', color='r')
_ = plt.hist(recon_energies[recon_energies<threshold], bins=100, histtype='step', color='g')
_ = plt.hist(sampled_energies[sampled_energies<threshold], bins=100, histtype='step', color='b')

plt.legend(('Test Data Potential', 'Reconstructed from Test', 'Sampled from Mixture Model'))
plt.xlabel('Energy (kJ/mol)')
plt.title('Comparison of Energies - GMM and NN')
plt.show()

In [ ]:
plt.clf()
_ = plt.hist(org_energies, bins=100, histtype='step', color='r')
plt.show()
plt.clf()
_ = plt.hist(recon_energies, bins=100, histtype='step', color='g')
plt.show()
plt.clf()
_ = plt.hist(sampled_energies, bins=100, histtype='step', color='b')
plt.show()

In [ ]:
experiment.train_summation(1e-1, 100, cutoff_epoch)
print('END TRAINING')
walltime = datetime.now() - start
print(f'Walltime per epoch was {walltime/n_epochs}')
experiment.save_loss_data()
experiment.write_model_to_ckpt()

In [ ]:
walltime = datetime.now() - start
print(f'Walltime per epoch was {walltime/n_epochs}')

In [ ]:
#Init Experiment
experiment = AutoEncoder_Experiment(json_fn)
experiment.restore_latest()
print(experiment.epoch)
experiment.train_nepochs_on_rmsd(200)
experiment.save_loss_data()

In [ ]:
walltime = datetime.now() - start
print(f'Walltime per epoch was {walltime/n_epochs}')

In [ ]:
# # RMSD BLOCK 2
# # Train until last 100 vals average less than predefined cutoff, always make sure we never train longer than cutoff_epoch
# begin_scaling_epoch = experiment.train_rmsd_threshold(nm_cutoff, num_move_ave, rmsd_cutoff)
# experiment.write_model_to_ckpt()
# print('END RMSD')

In [ ]:
begin_scaling_epoch = experiment.epoch
print(experiment.epoch)

In [ ]:
# SCALE IN POTENTIAL BLOCK
end_scaling_epoch = experiment.train_scaling_potential(cutoff_epoch)
print('END SCALING POTENTIAL')
experiment.save_loss_data()
experiment.write_model_to_ckpt()

In [ ]:
experiment.train_summation(1e-1, 100, cutoff_epoch)
print('END TRAINING')
experiment.save_loss_data()
experiment.write_model_to_ckpt()
experiment.graph_losses(begin_scaling_epoch=begin_scaling_epoch, end_scaling_epoch=end_scaling_epoch)

In [ ]:
experiment.train_summation(1e-2, 100, cutoff_epoch)
print('END TRAINING')
experiment.save_loss_data()
experiment.write_model_to_ckpt()
experiment.graph_losses(begin_scaling_epoch=begin_scaling_epoch, end_scaling_epoch=end_scaling_epoch)